# Olist OLAP and ML feature exploration

Build the warehouse first from the project root with `uv run kddm-lab build`. The database and outputs are local and are not committed.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt

database = Path("data/olist.duckdb")
connection = duckdb.connect(str(database), read_only=True)

## OLAP: monthly revenue and category roll-up

In [ ]:
monthly_sales = connection.execute(
    "SELECT * FROM monthly_sales ORDER BY year, month"
).fetchdf()
monthly_sales.tail()

In [ ]:
monthly_sales = monthly_sales.assign(
    month_label=monthly_sales["year"].astype(str)
    + "-"
    + monthly_sales["month"].astype(str).str.zfill(2)
)
monthly_sales.plot(
    x="month_label",
    y="gross_merchandise_value",
    kind="line",
    figsize=(12, 4),
    title="Monthly gross merchandise value",
    xlabel="Purchase month",
    ylabel="GMV",
    legend=False,
)
plt.xticks(rotation=60)
plt.tight_layout()

In [ ]:
connection.execute(
    "SELECT * FROM category_sales ORDER BY gross_merchandise_value DESC LIMIT 10"
).fetchdf()

## ML-ready customer features

These descriptive features use the entire dataset through its latest purchase date. For supervised learning, choose a historical cutoff and calculate labels only from a separate future outcome window; otherwise the full-period recency and aggregates can leak future information.

In [ ]:
customer_features = connection.execute(
    "SELECT * FROM ml_customer_features ORDER BY total_gmv DESC LIMIT 10"
).fetchdf()
customer_features

In [ ]:
connection.close()